In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_PATH = Path("../data/raw/creditcard.csv")
PROCESSED_DIR = Path("../data/processed") 
SAMPLE_DIR = Path("../data/sample")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True) #保证这两个directory存在，已经存在也没事
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(RAW_PATH)

print(df.shape)
df.head() #0-noraml, 1-fraud

(284807, 31)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


fraud vs. normal transaction counts

In [3]:
df["Class"].value_counts()

Class
0    284315
1       492
Name: count, dtype: int64

fraud rate

In [4]:
fraud_count = df["Class"].sum()
total_count = len(df)
fraud_rate = fraud_count / total_count

print("Total transactions:", total_count)
print("Fraud transactions:", fraud_count)
print("Fraud rate:", fraud_rate)

Total transactions: 284807
Fraud transactions: 492
Fraud rate: 0.001727485630620034


In [5]:
sample = df.sample(1000, random_state=42)
sample.to_csv("../data/sample/sample_transactions.csv", index=False)

sample.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
43428,41505.0,-16.526507,8.584972,-18.649853,9.505594,-13.793819,-2.832404,-16.701694,7.517344,-8.507059,...,1.190739,-1.127670,-2.358579,0.673461,-1.413700,-0.462762,-2.018575,-1.042804,364.19,1
49906,44261.0,0.339812,-2.743745,-0.134070,-1.385729,-1.451413,1.015887,-0.524379,0.224060,0.899746,...,-0.213436,-0.942525,-0.526819,-1.156992,0.311211,-0.746647,0.040996,0.102038,520.12,0
29474,35484.0,1.399590,-0.590701,0.168619,-1.029950,-0.539806,0.040444,-0.712567,0.002299,-0.971747,...,0.102398,0.168269,-0.166639,-0.810250,0.505083,-0.232340,0.011409,0.004634,31.00,0
276481,167123.0,-0.432071,1.647895,-1.669361,-0.349504,0.785785,-0.630647,0.276990,0.586025,-0.484715,...,0.358932,0.873663,-0.178642,-0.017171,-0.207392,-0.157756,-0.237386,0.001934,1.50,0
278846,168473.0,2.014160,-0.137394,-1.015839,0.327269,-0.182179,-0.956571,0.043241,-0.160746,0.363241,...,-0.238644,-0.616400,0.347045,0.061561,-0.360196,0.174730,-0.078043,-0.070571,0.89,0


Summary table

In [6]:
kpi_summary = pd.DataFrame({
    "metric": ["total_transactions", "fraud_transactions", "fraud_rate"],
    "value": [total_count, fraud_count, fraud_rate]
})

kpi_summary.to_csv("../data/processed/kpi_summary.csv", index=False)
kpi_summary

,metric,value
0,total_transactions,284807.000000
1,fraud_transactions,492.000000
2,fraud_rate,0.001727


In [7]:
amount_summary = df.groupby("Class")["Amount"].agg(["count", "mean", "median", "max", "sum"]).reset_index()

amount_summary["Class"] = amount_summary["Class"].replace({
    0: "Normal",
    1: "Fraud"
})

amount_summary.to_csv("../data/processed/amount_summary_by_class.csv", index=False)
amount_summary

,Class,count,mean,median,max,sum
0,Normal,284315,88.291022,22.00,25691.16,25102462.04
1,Fraud,492,122.211321,9.25,2125.87,60127.97


Hourly fraud rate

In [8]:
df["hour"] = (df["Time"] // 3600) % 24

hourly_summary = df.groupby("hour").agg(
    total_transactions=("Class", "count"),
    fraud_transactions=("Class", "sum")
).reset_index()

hourly_summary["fraud_rate"] = hourly_summary["fraud_transactions"] / hourly_summary["total_transactions"]

hourly_summary.to_csv("../data/processed/hourly_fraud_rate.csv", index=False)
hourly_summary.head()

,hour,total_transactions,fraud_transactions,fraud_rate
0,0.0,7695,6,0.000780
1,1.0,4220,10,0.002370
2,2.0,3328,57,0.017127
3,3.0,3492,17,0.004868
4,4.0,2209,23,0.010412


In [9]:
import os

print(os.listdir("../data/processed"))
print(os.listdir("../data/sample"))

['cost_analysis.csv', 'random_forest_summary.csv', 'model_comparison.csv', 'threshold_analysis.csv', 'kpi_summary.csv', 'hourly_fraud_rate.csv', 'logistic_regression_summary.csv', 'amount_summary_by_class.csv']
['sample_transactions.csv', 'scored_transactions_sample.csv']
